# Rodada confirmatória

O notebook lê os oito arquivos da rodada de 2026, em `reports/confirmatorio/`: `x7a.json`, `x7b.json` e `x7a_s1.json` a
`x7a_s6.json`. No X7a, o treino vai de 2019 a 2024, a calibração usa 2025 e o teste, janeiro a março de 2026, com as
competências até jul/2026; as seis sensibilidades rodam sobre ele. O X7b testa no mesmo trimestre, mas treina e calibra
três meses mais cedo, com a calibração de out/2024 a set/2025. Falta o X7c, que testa as internações do 2º trimestre de
2026 e espera as competências até out/2026.

In [1]:
import json

import pandas as pd

from triagem_baixo_risco import config
from triagem_baixo_risco.modeling.predict import exact_ci

CONFIRMATORIO = config.REPORTS_DIR / "confirmatorio"
RODADAS = ["x7a", "x7b", "x7a_s1", "x7a_s2", "x7a_s3", "x7a_s4", "x7a_s5", "x7a_s6"]
pd.set_option("display.max_colwidth", None)


def pct(x, casas=2):
    return f"{100 * x:.{casas}f}%".replace(".", ",")


def num(x, casas=0):
    # 1234567.891 com casas=2 vira "1.234.567,89"
    return f"{x:,.{casas}f}".replace(",", " ").replace(".", ",").replace(" ", ".")


def pv(p):
    return "< 0,0001" if p < 0.0001 else num(p, 4)


def pontos(x):
    return f"{100 * x:+.1f}".replace(".", ",")


def com_ic(d):
    return f"{pontos(d['diferenca'])} ({pontos(d['ic'][0])} a {pontos(d['ic'][1])})"


def decisao(refutada):
    return {True: "refutada", False: "não refutada", None: "sem previsão"}[refutada]


def decisao_h2(h2):
    texto = f"{h2['previsao']}, razão {num(h2['razao'], 4)}"
    return texto if h2["refutada"] is None else f"{texto}: {decisao(h2['refutada'])}"


res = {r: json.loads((CONFIRMATORIO / f"{r}.json").read_text()) for r in RODADAS}

## Hipóteses

O nível principal é α = 0,05, e o de 0,10 vem ao lado. Para H2, o que conta é a prevalência de óbito do teste dividida
pela da parte B: até 1,0, G2 deve valer, e de 1,2 para cima deve falhar. No meio disso, H2 não prevê nada. Quanto a H5,
só o X7c a testa de forma confirmatória; no X7a ela entra como descrição, e o X7b não a avalia.

In [2]:
ESCOPO_H5 = {"descritiva": "descritiva", "nao_avaliada": "não avaliada"}


def hipoteses(r):
    linhas = {}
    for alfa, a in r["alfas"].items():
        linhas[num(float(alfa), 2)] = {
            "H1": decisao(a["h1"]["refutada"]),
            "H1b": decisao(a["h1b"]["refutada"]),
            "H2": decisao_h2(a["h2"]),
            "H4": decisao(a["h4"]["refutada"]),
            "H5": ESCOPO_H5.get(r["escopo_h5"]) or decisao(a["h5"]["refutada"]),
        }
    return pd.DataFrame(linhas).T


pd.concat({"X7a": hipoteses(res["x7a"]), "X7b": hipoteses(res["x7b"])}, names=["rodada", "α"])

H1           H1b                                   H2  \
rodada α                                                                       
X7a    0,05  não refutada  não refutada  G2 vale, razão 0,9224: não refutada   
       0,10  não refutada  não refutada  G2 vale, razão 0,9224: não refutada   
X7b    0,05  não refutada  não refutada  G2 vale, razão 0,9092: não refutada   
       0,10  não refutada  não refutada  G2 vale, razão 0,9092: não refutada   

                       H4            H5  
rodada α                                 
X7a    0,05  não refutada    descritiva  
       0,10  não refutada    descritiva  
X7b    0,05  não refutada  não avaliada  
       0,10  não refutada  não avaliada

Nenhuma hipótese foi refutada, nem no X7a nem no X7b, com nenhum dos dois α. A razão de prevalência ficou em 0,9224 no
X7a e em 0,9092 no X7b, então H2 previa que G2 valesse, e ela valeu.

## Garantias e utilidade

Na tabela, o IC exato da FDL é o de Clopper-Pearson, e o de hospitais vem do bootstrap que reamostra hospitais. As duas
últimas linhas são as diferenças de fração liberada de H4, em pontos percentuais, com o IC de hospitais, que é o que
decide.

In [3]:
def garantias(r):
    linhas = {}
    for alfa, a in r["alfas"].items():
        m, u = a["metricas"], a["utilidade"]
        lo, hi = exact_ci(m["fdl_k"], m["fdl_n"])
        boot = u["bootstrap_hgb"]["fdl"]["hospitais"]
        linhas[num(float(alfa), 2)] = {
            "liberadas": f"{num(m['tol_n'])} ({pct(m['fracao_liberada'], 1)})",
            "adultos não obstétricos": pct(u["hgb"]["fracao_liberada_adultos"], 1),
            "óbitos liberados": f"{m['fdl_k']} de {num(m['fdl_n'])}",
            "FDL": pct(m["fdl"]),
            "IC exato": f"{pct(lo, 1)} a {pct(hi, 1)}",
            "IC, hospitais": f"{pct(boot[0], 1)} a {pct(boot[1], 1)}",
            "p G1": pv(m["p_g1"]),
            "TOL": pct(m["tol"]),
            "r̂": pct(m["r_hat"]),
            "p G2": pv(m["p_g2"]),
            "contra a regra sem modelo": com_ic(u["diferencas"]["regra"]),
            "contra a logística de 4 variáveis": com_ic(u["diferencas"]["logistica_simples"]),
        }
    return pd.DataFrame(linhas)


pd.concat({"X7a": garantias(res["x7a"]), "X7b": garantias(res["x7b"])}, axis=1, names=["rodada", "α"])

rodada                                               X7a  \
α                                                   0,05   
liberadas                                 23.769 (64,9%)   
adultos não obstétricos                            54,0%   
óbitos liberados                             59 de 1.345   
FDL                                                4,39%   
IC exato                                     3,4% a 5,6%   
IC, hospitais                                2,9% a 6,0%   
p G1                                              0,8066   
TOL                                                0,25%   
r̂                                                 0,35%   
p G2                                              0,9969   
contra a regra sem modelo          +17,8 (+14,4 a +21,3)   
contra a logística de 4 variáveis   +10,1 (+7,3 a +13,9)   

rodada                                                    \
α                                                   0,10   
liberadas                                 26.062 (71,2%)   
adultos não obstétricos                            61,8%   
óbitos liberados                            108 de 1.345   
FDL                                                8,03%   
IC exato                                     6,6% a 9,6%   
IC, hospitais                               6,3% a 10,5%   
p G1                                              0,9813   
TOL                                                0,41%   
r̂                                                 0,61%   
p G2                                              1,0000   
contra a regra sem modelo          +17,8 (+14,4 a +21,3)   
contra a logística de 4 variáveis    +6,1 (+2,6 a +10,5)   

rodada                                               X7b  \
α                                                   0,05   
liberadas                                 24.015 (65,6%)   
adultos não obstétricos                            54,9%   
óbitos liberados                             63 de 1.345   
FDL                                                4,68%   
IC exato                                     3,6% a 6,0%   
IC, hospitais                                3,4% a 6,6%   
p G1                                              0,6658   
TOL                                                0,26%   
r̂                                                 0,35%   
p G2                                              0,9912   
contra a regra sem modelo          +17,9 (+14,6 a +21,3)   
contra a logística de 4 variáveis    +9,8 (+7,0 a +13,6)   

rodada                                                    
α                                                   0,10  
liberadas                                 26.294 (71,8%)  
adultos não obstétricos                            62,6%  
óbitos liberados                            120 de 1.345  
FDL                                                8,92%  
IC exato                                    7,5% a 10,6%  
IC, hospitais                               7,0% a 11,6%  
p G1                                              0,8665  
TOL                                                0,46%  
r̂                                                 0,61%  
p G2                                              0,9997  
contra a regra sem modelo          +17,9 (+14,6 a +21,3)  
contra a logística de 4 variáveis     +5,2 (+2,0 a +9,6)

Com α = 0,05, o X7a liberou 23.769 internações, 64,9% do trimestre, e entre elas estavam 59 dos 1.345 óbitos do teste. A
FDL fica em 4,39%, com p = 0,8066 no teste de violação de G1. Ainda assim, os intervalos são largos: de 3,4% a 5,6% pelo
IC exato e de 2,9% a 6,0% pelo de hospitais. Como os dois passam de 5%, não dá para descartar que a FDL real fique um
pouco acima da meta. A TOL, taxa de óbito entre as liberadas, ficou em 0,25%, abaixo do r̂ de 0,35%.

Em H4, 54,0% das internações de adultos não obstétricos foram liberadas, acima dos 40% pedidos. O boosting liberou 17,8
pontos a mais que a regra sem modelo e 10,1 a mais que a logística de 4 variáveis, com IC acima de zero nas duas
comparações. O X7b ficou perto disso em tudo: liberou 65,6% das internações, e 63 dos 1.345 óbitos ficaram entre elas,
uma FDL de 4,68%.

In [4]:
def sem_surto(r):
    m = r["alfas"]["0.05"]["metricas"]
    return {
        "CID em surto": ", ".join(m["cids_em_surto"]) or "-",
        "óbitos com o CID": m["fdl_n"] - m["fdl_sem_surto_n"],
        "liberados com o CID": m["fdl_k"] - m["fdl_sem_surto_k"],
        "FDL sem o CID": pct(m["fdl_sem_surto"]),
        "p G1 sem o CID": pv(m["p_g1_sem_surto"]),
    }


pd.DataFrame({"X7a": sem_surto(res["x7a"]), "X7b": sem_surto(res["x7b"])})

,X7a,X7b
CID em surto,B01,B01
óbitos com o CID,15,15
liberados com o CID,0,0
FDL sem o CID,"4,44%","4,74%"
p G1 sem o CID,"0,7853","0,6375"


No teste, o único código em surto foi o B01, a varicela, com 15 óbitos, nenhum liberado. Sem eles, a FDL do X7a vai a
4,44%, e H1b também não é refutada.

## Desempenho

In [5]:
def intervalo(par):
    return f"{num(par[0], 3)} a {num(par[1], 3)}"


def desempenho(r):
    m = r["alfas"]["0.05"]["metricas"]
    boot = r["alfas"]["0.05"]["utilidade"]["bootstrap_hgb"]
    return {
        "internações": num(r["n_teste"]),
        "óbitos": num(r["obitos_teste"]),
        "prevalência": pct(m["prevalencia"]),
        "razão de prevalência": num(m["razao_prevalencia"], 4),
        "AUROC": f"{num(m['auroc'], 3)} ({intervalo(m['auroc_ic'])})",
        "AUROC, IC de hospitais": intervalo(boot["auroc"]["hospitais"]),
        "O/E": f"{num(m['oe'], 3)} ({intervalo(m['oe_ic'])})",
        "O/E, IC de hospitais": intervalo(boot["oe"]["hospitais"]),
        "intercepto": f"{num(m['intercepto'], 3)} ({intervalo(m['intercepto_ic'])})",
        "slope": f"{num(m['slope'], 3)} ({intervalo(m['slope_ic'])})",
        "ICI": num(m["ici"], 4),
    }


pd.DataFrame({"X7a": desempenho(res["x7a"]), "X7b": desempenho(res["x7b"])})

,X7a,X7b
internações,36.621,36.621
óbitos,1.345,1.345
prevalência,"3,67%","3,67%"
razão de prevalência,"0,9224","0,9092"
AUROC,"0,898 (0,887 a 0,909)","0,898 (0,887 a 0,909)"
"AUROC, IC de hospitais","0,880 a 0,914","0,880 a 0,915"
O/E,"0,934 (0,885 a 0,985)","0,916 (0,868 a 0,966)"
"O/E, IC de hospitais","0,862 a 1,011","0,845 a 0,990"
intercepto,"-0,084 (-0,143 a -0,025)","-0,107 (-0,166 a -0,048)"
slope,"0,959 (0,911 a 1,007)","0,983 (0,933 a 1,032)"


As duas rodadas tiveram a mesma AUROC, 0,898. No X7a, o O/E de 0,934 e o intercepto negativo mostram o modelo, calibrado
com 2025, ainda prevendo mais óbitos do que houve no começo de 2026. Mas o IC do O/E pelo bootstrap de hospitais, de
0,862 a 1,011, inclui 1.

## Faixas etárias

O Mondrian cruzado calcula o corte de cada faixa com os óbitos dela na parte B. Como H5 é descritiva no X7a, a coluna do
teste de H5 só diz se a faixa entraria nele, o que exige pelo menos 100 óbitos no teste.

In [6]:
def faixas(a):
    tabela = {}
    for faixa, m in a["cruzado_por_faixa"].items():
        tabela[faixa] = {
            "internações": num(m["n"]),
            "liberadas": pct(m["fracao_liberada"], 1),
            "óbitos liberados": f"{m['fdl_k']} de {num(m['fdl_n'])}",
            "FDL": pct(m["fdl"]),
            "p G1": pv(m["p_g1"]),
            "entraria no teste de H5": "não" if a["h5"]["faixas"][faixa].get("descritiva") else "sim",
            "TOL": pct(m["tol"]),
            "p G2": pv(m["p_g2"]),
        }
    return pd.DataFrame(tabela).T.rename_axis("faixa")


pd.concat({num(float(alfa), 2): faixas(a) for alfa, a in res["x7a"]["alfas"].items()}, names=["α"])

internações liberadas óbitos liberados     FDL    p G1  \
α    faixa                                                          
0,05 0-17        5.048     59,9%          0 de 23   0,00%  1,0000   
     18-39      10.767     77,4%          3 de 68   4,41%  0,5754   
     40-59       8.467     62,5%        13 de 241   5,39%  0,4073   
     60-79       8.791     44,9%        28 de 549   5,10%  0,4605   
     80+         3.548     26,0%        30 de 464   6,47%  0,1212   
0,10 0-17        5.048     77,2%          1 de 23   4,35%  0,8544   
     18-39      10.767     85,2%          7 de 68  10,29%  0,5057   
     40-59       8.467     68,0%        26 de 241  10,79%  0,3850   
     60-79       8.791     54,5%        48 de 549   8,74%  0,8041   
     80+         3.548     34,6%        50 de 464  10,78%  0,3188   

           entraria no teste de H5    TOL      p G2  
α    faixa                                           
0,05 0-17                      não  0,00%    1,0000  
     18-39                     não  0,04%    1,0000  
     40-59                     sim  0,25%    0,9529  
     60-79                     sim  0,71%    0,0011  
     80+                       sim  3,25%  < 0,0001  
0,10 0-17                      não  0,03%    1,0000  
     18-39                     não  0,08%    1,0000  
     40-59                     sim  0,45%    0,9698  
     60-79                     sim  1,00%    0,0017  
     80+                       sim  4,08%  < 0,0001

Com α = 0,05, nenhuma das três faixas com pelo menos 100 óbitos teria violado G1: o menor p foi 0,1212, dos 80 anos ou
mais, com FDL de 6,47%. As faixas de 0 a 17 e de 18 a 39 anos, com 23 e 68 óbitos, ficam só descritas.

Nos 80 anos ou mais, porém, 3,25% das internações liberadas terminaram em óbito, e 30 dos 464 óbitos da faixa foram
liberados. G2 nunca foi prometida por faixa, e o r̂ vale para a regra toda, mas quem usasse a regra num serviço de
idosos precisaria saber desse número.

## Sensibilidades

In [7]:
NOMES = {
    "s1": "S1, isotônica",
    "s2": "S2, sem o CID de 3 caracteres",
    "s3": "S3, sem as censuradas",
    "s4": "S4, coorte pela competência",
    "s5": "S5, óbito por episódio",
    "s6": "S6, sem hospital e município de atendimento",
}
FALHAS_H4 = {
    "fracao_todas": "fração de todas",
    "fracao_adultos": "fração de adultos",
    "vs_regra": "contra a regra sem modelo",
    "vs_logistica_simples": "contra a logística de 4 variáveis",
}
linhas = {}
for s, nome in NOMES.items():
    r = res[f"x7a_{s}"]
    for alfa, a in r["alfas"].items():
        m = a["metricas"]
        falhas = [FALHAS_H4[f] for f, x in a["h4"]["falhas"].items() if x]
        linhas[(nome, num(float(alfa), 2))] = {
            "internações": num(r["n_teste"]),
            "óbitos liberados": f"{m['fdl_k']} de {num(m['fdl_n'])}",
            "FDL": pct(m["fdl"]),
            "p G1": pv(m["p_g1"]),
            "H1": decisao(a["h1"]["refutada"]),
            "H1b": decisao(a["h1b"]["refutada"]),
            "H2": decisao_h2(a["h2"]),
            "H4": decisao(a["h4"]["refutada"]) + (f" ({', '.join(falhas)})" if falhas else ""),
            "contra a logística de 4 variáveis": com_ic(a["utilidade"]["diferencas"]["logistica_simples"]),
        }
pd.DataFrame(linhas).T.rename_axis(["sensibilidade", "α"])

internações óbitos liberados  \
sensibilidade                               α                                   
S1, isotônica                               0,05      36.621      29 de 1.345   
                                            0,10      36.621     108 de 1.345   
S2, sem o CID de 3 caracteres               0,05      36.621      59 de 1.345   
                                            0,10      36.621     107 de 1.345   
S3, sem as censuradas                       0,05      32.428      73 de 1.345   
                                            0,10      32.428     146 de 1.345   
S4, coorte pela competência                 0,05      35.309      66 de 1.360   
                                            0,10      35.309     137 de 1.360   
S5, óbito por episódio                      0,05      33.779      63 de 1.339   
                                            0,10      33.779     134 de 1.339   
S6, sem hospital e município de atendimento 0,05      36.621      64 de 1.345   
                                            0,10      36.621     124 de 1.345   

                                                     FDL    p G1  \
sensibilidade                               α                      
S1, isotônica                               0,05   2,16%  1,0000   
                                            0,10   8,03%  0,9813   
S2, sem o CID de 3 caracteres               0,05   4,39%  0,8066   
                                            0,10   7,96%  0,9847   
S3, sem as censuradas                       0,05   5,43%  0,2757   
                                            0,10  10,86%  0,2000   
S4, coorte pela competência                 0,05   4,85%  0,5786   
                                            0,10  10,07%  0,4746   
S5, óbito por episódio                      0,05   4,71%  0,6654   
                                            0,10  10,01%  0,4975   
S6, sem hospital e município de atendimento 0,05   4,76%  0,6281   
                                            0,10   9,22%  0,7893   

                                                            H1           H1b  \
sensibilidade                               α                                  
S1, isotônica                               0,05  não refutada  não refutada   
                                            0,10  não refutada  não refutada   
S2, sem o CID de 3 caracteres               0,05  não refutada  não refutada   
                                            0,10  não refutada  não refutada   
S3, sem as censuradas                       0,05  não refutada  não refutada   
                                            0,10  não refutada  não refutada   
S4, coorte pela competência                 0,05  não refutada  não refutada   
                                            0,10  não refutada  não refutada   
S5, óbito por episódio                      0,05  não refutada  não refutada   
                                            0,10  não refutada  não refutada   
S6, sem hospital e município de atendimento 0,05  não refutada  não refutada   
                                            0,10  não refutada  não refutada   

                                                                                   H2  \
sensibilidade                               α                                           
S1, isotônica                               0,05  G2 vale, razão 0,9224: não refutada   
                                            0,10  G2 vale, razão 0,9224: não refutada   
S2, sem o CID de 3 caracteres               0,05  G2 vale, razão 0,9224: não refutada   
                                            0,10  G2 vale, razão 0,9224: não refutada   
S3, sem as censuradas                       0,05  G2 vale, razão 0,9758: não refutada   
                                            0,10  G2 vale, razão 0,9758: não refutada   
S4, coorte pela competência                 0,05           sem previsão, razão 1,0004   
                                       

Com α = 0,05, nenhuma sensibilidade refutou hipótese. Com α = 0,10, a S2 refutou H4: sem o CID de 3 caracteres, o
boosting liberou 3,0 pontos a mais que a logística de 4 variáveis, e o IC, de -0,3 a 7,3, inclui zero. A S4, que monta a
coorte pela competência, levou a razão de prevalência a 1,0004: passou de 1,0 por muito pouco, mas o bastante para cair
na faixa em que H2 não prevê nada.

As outras mudam os números sem mudar decisão. Com a isotônica (S1), a regra de α = 0,05 liberou só 29 óbitos, 2,16%. Sem
as censuradas (S3), o teste fica com 32.428 internações e a FDL sobe para 5,43%, com p = 0,2757.

## A correção da S2 e da S6

A S2 e a S6 rodaram depois de uma correção na logística completa, que passou a usar as variáveis de cada sensibilidade,
como o boosting já fazia; antes, ela montava sempre a mesma lista de variáveis, com o CID de 3 caracteres, justamente a
coluna que a S2 tira. A logística completa é só descritiva: H1 a H5 se decidem pelo boosting, pela regra sem modelo e
pela logística de 4 variáveis, em que a correção não mexe. Por isso ela não muda decisão nenhuma, nem a refutação de H4
pela S2 com α = 0,10, que vem da diferença contra a logística de 4 variáveis.